# 🏥 Insulin Dosage Prediction - Final Clinical Pipeline
**XGBoost GPU + Clinical Safety Net V2.0**

---

## 📦 Cell 1: Setup & Libraries

In [ ]:
# Install
!pip install polars xgboost -q

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# Imports
import polars as pl
import numpy as np
import pandas as pd
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_error, mean_absolute_error
import math
import time
import os
import gc

print(f"✅ Polars: {pl.__version__}")
print(f"✅ XGBoost: {xgb.__version__}")

# GPU Check
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## ⚙️ Cell 2: Configuration

In [ ]:
# ==============================================================================
# CONFIGURATION
# ==============================================================================

BASE_PATH = "/content/drive/MyDrive/DiabetesKDS/data"

RAW_TEST_FILE = f"{BASE_PATH}/raw/metabonet_public_test.parquet"
PROCESSED_TEST_FILE = f"{BASE_PATH}/processed/test_ready_for_xgboost.parquet"
TRAIN_FILE = f"{BASE_PATH}/processed/train_augmented_fixed.parquet"
MODEL_OUTPUT = f"{BASE_PATH}/processed/xgboost_final_model.json"

# FEATURES - STRICTLY NUMERICAL (No strings!)
FEATURES = [
    'CGM', 'carbs', 'time_sin', 'time_cos', 'is_weekend',
    'cgm_delta_5min', 'cgm_delta_15min', 'basal', 'age', 'weight',
    'gender_encoded', 'insulin_type_encoded'
]
TARGET = 'bolus'

# Encoding Maps
GENDER_MAP = {'Male': 0, 'Female': 1, 'M': 0, 'F': 1}
INSULIN_MAP = {
    'Humalog': 3, 'Novolog': 6, 'NovoLog': 6, 'Fiasp': 2,
    'Apidra': 0, 'Regular': 8, 'Humalog (Lispro)': 4
}

# XGBoost
PARAMS = {
    'objective': 'reg:squarederror',
    'eval_metric': ['rmse', 'mae'],
    'tree_method': 'hist',
    'device': 'cuda',
    'max_depth': 12,
    'learning_rate': 0.1,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'scale_pos_weight': 50,
    'seed': 42,
}
NUM_ROUNDS = 500

# Clinical Thresholds
CGM_CRITICAL = 250  # mg/dL
CGM_HIGH = 160      # mg/dL
DOSE_THRESHOLD = 0.5

print("=" * 60)
print("🏥 Clinical ML Pipeline - Configuration")
print("=" * 60)
print(f"📊 Features: {len(FEATURES)} numerical columns")
print(f"🎯 Target: {TARGET}")

---
## 🟢 Cell 3: Step 1 - Robust Test Preprocessing

In [ ]:
print("=" * 60)
print("🟢 STEP 1: TEST DATA PREPROCESSING")
print("=" * 60)

# Load
df = pl.read_parquet(RAW_TEST_FILE)
print(f"✅ Loaded: {df.shape[0]:,} × {df.shape[1]}")

# Date features
if 'date' in df.columns:
    df = df.with_columns(pl.col('date').cast(pl.Datetime))
    df = df.with_columns([
        (2 * math.pi * pl.col('date').dt.hour() / 24).sin().alias('time_sin'),
        (2 * math.pi * pl.col('date').dt.hour() / 24).cos().alias('time_cos'),
        (pl.col('date').dt.weekday() >= 5).cast(pl.Int32).alias('is_weekend'),
    ])
print("✓ time_sin, time_cos, is_weekend")

# Lag features
if all(c in df.columns for c in ['id', 'date', 'CGM']):
    df = df.sort(['id', 'date'])
    df = df.with_columns([
        (pl.col('CGM') - pl.col('CGM').shift(1).over('id')).fill_null(0.0).alias('cgm_delta_5min'),
        (pl.col('CGM') - pl.col('CGM').shift(3).over('id')).fill_null(0.0).alias('cgm_delta_15min'),
    ])
print("✓ cgm_delta_5min, cgm_delta_15min")

# Encoding
if 'gender' in df.columns:
    df = df.with_columns(
        pl.col('gender').map_elements(lambda x: GENDER_MAP.get(x, -1) if x else -1, return_dtype=pl.Int32).alias('gender_encoded')
    )
else:
    df = df.with_columns(pl.lit(-1).alias('gender_encoded'))

if 'insulin_type_bolus' in df.columns:
    df = df.with_columns(
        pl.col('insulin_type_bolus').map_elements(lambda x: INSULIN_MAP.get(x, -1) if x else -1, return_dtype=pl.Int32).alias('insulin_type_encoded')
    )
else:
    df = df.with_columns(pl.lit(-1).alias('insulin_type_encoded'))
print("✓ gender_encoded, insulin_type_encoded")

# Imputation
for col in ['weight', 'basal', 'age']:
    if col in df.columns:
        df = df.with_columns(pl.col(col).fill_null(-1.0).cast(pl.Float32))
    else:
        df = df.with_columns(pl.lit(-1.0).cast(pl.Float32).alias(col))
print("✓ weight, basal, age imputed")

# Select ONLY numerical features
cols = [f for f in FEATURES if f in df.columns]
has_target = TARGET in df.columns
if has_target:
    cols.append(TARGET)

df_test = df.select(cols)
for c in df_test.columns:
    df_test = df_test.with_columns(pl.col(c).cast(pl.Float32))

# Save
df_test.write_parquet(PROCESSED_TEST_FILE)
print(f"\n💾 Saved: {PROCESSED_TEST_FILE}")
print(f"📊 Shape: {df_test.shape}")
print(f"📊 Dtypes: {df_test.dtypes}")

del df
gc.collect()
print("\n✅ STEP 1 COMPLETE")

---
## 🔴 Cell 4: Step 2 - Full Training (GPU)

In [ ]:
print("=" * 60)
print("🔴 STEP 2: FULL TRAINING (NO SPLIT)")
print("=" * 60)

# Load training data
df_train = pl.read_parquet(TRAIN_FILE)
print(f"✅ Train loaded: {df_train.shape[0]:,} × {df_train.shape[1]}")

# Encode training data
if 'gender' in df_train.columns and 'gender_encoded' not in df_train.columns:
    if str(df_train['gender'].dtype).startswith('String') or df_train['gender'].dtype == pl.Utf8:
        df_train = df_train.with_columns(
            pl.col('gender').map_elements(lambda x: GENDER_MAP.get(x, -1) if x else -1, return_dtype=pl.Int32).alias('gender_encoded')
        )
    else:
        df_train = df_train.with_columns(pl.col('gender').fill_null(-1).cast(pl.Int32).alias('gender_encoded'))

if 'insulin_type_bolus' in df_train.columns and 'insulin_type_encoded' not in df_train.columns:
    if str(df_train['insulin_type_bolus'].dtype).startswith('String') or df_train['insulin_type_bolus'].dtype == pl.Utf8:
        df_train = df_train.with_columns(
            pl.col('insulin_type_bolus').map_elements(lambda x: INSULIN_MAP.get(x, -1) if x else -1, return_dtype=pl.Int32).alias('insulin_type_encoded')
        )
    else:
        df_train = df_train.with_columns(pl.col('insulin_type_bolus').fill_null(-1).cast(pl.Int32).alias('insulin_type_encoded'))

for col in ['weight', 'basal', 'age']:
    if col in df_train.columns:
        df_train = df_train.with_columns(pl.col(col).fill_null(-1.0).cast(pl.Float32))

# Prepare training arrays
train_feats = [f for f in FEATURES if f in df_train.columns]
X_train = df_train.select(train_feats).to_numpy().astype(np.float32)
y_train = df_train.select(TARGET).to_numpy().ravel().astype(np.float32)

print(f"🏋️ X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"   Positive: {(y_train > 0).sum():,} ({(y_train > 0).mean()*100:.1f}%)")

del df_train
gc.collect()

# Prepare test arrays
test_feats = [f for f in train_feats if f in df_test.columns]
X_test = df_test.select(test_feats).to_numpy().astype(np.float32)
y_test_raw = df_test.select(TARGET).to_numpy().ravel().astype(np.float32) if has_target else None

print(f"📊 X_test: {X_test.shape}")

# DMatrix
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=test_feats)
dtest = xgb.DMatrix(X_test, feature_names=test_feats)

# Train
print(f"\n🚀 Training XGBoost ({NUM_ROUNDS} rounds)...")
t0 = time.time()
model = xgb.train(PARAMS, dtrain, NUM_ROUNDS, evals=[(dtrain, 'train')], verbose_eval=100)
print(f"\n✅ Done in {time.time()-t0:.1f}s")

# Save
model.save_model(MODEL_OUTPUT)
print(f"💾 Model saved: {MODEL_OUTPUT}")

---
## 📊 Cell 5: Advanced Evaluation & Clinical Safety Net V2.0

In [ ]:
print("=" * 60)
print("📊 CLINICAL EVALUATION + SAFETY NET V2.0")
print("=" * 60)

# Predict
y_pred_raw = model.predict(dtest)
y_pred_raw = np.maximum(y_pred_raw, 0)  # Clip negatives

if has_target:
    # ========================================
    # NaN SAFETY - Filter out NaN targets
    # ========================================
    valid_mask = ~np.isnan(y_test_raw)
    n_nan = (~valid_mask).sum()
    print(f"\n⚠️ NaN targets filtered: {n_nan:,} / {len(y_test_raw):,}")
    
    y_test = y_test_raw[valid_mask]
    y_pred = y_pred_raw[valid_mask]
    X_test_valid = X_test[valid_mask]
    
    # Get CGM values for safety net
    cgm_idx = test_feats.index('CGM') if 'CGM' in test_feats else None
    cgm_values = X_test_valid[:, cgm_idx] if cgm_idx is not None else None
    
    # ========================================
    # GLOBAL METRICS
    # ========================================
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    
    print(f"\n" + "-" * 50)
    print(f"📈 GLOBAL METRICS (n={len(y_test):,})")
    print(f"-" * 50)
    print(f"   RMSE: {rmse:.4f}")
    print(f"   MAE:  {mae:.4f}")
    
    # ========================================
    # POSITIVE-ONLY METRICS
    # ========================================
    pos_mask = y_test > 0
    if pos_mask.sum() > 0:
        pos_mae = mean_absolute_error(y_test[pos_mask], y_pred[pos_mask])
        print(f"\n" + "-" * 50)
        print(f"⭐ POSITIVE-ONLY (n={pos_mask.sum():,})")
        print(f"-" * 50)
        print(f"   MAE: {pos_mae:.4f}  ← KEY METRIC")
    
    # ========================================
    # 🏥 CLINICAL SAFETY NET V2.0
    # ========================================
    print(f"\n" + "=" * 50)
    print(f"🏥 CLINICAL SAFETY NET V2.0")
    print(f"=" * 50)
    
    # Identify missed doses
    missed_mask = (y_test > DOSE_THRESHOLD) & (y_pred < DOSE_THRESHOLD)
    n_missed_initial = missed_mask.sum()
    initial_error_rate = n_missed_initial / pos_mask.sum() * 100 if pos_mask.sum() > 0 else 0
    
    print(f"\n📊 Initial Analysis:")
    print(f"   Total positive cases: {pos_mask.sum():,}")
    print(f"   Missed doses (pred<0.5 when true>0.5): {n_missed_initial:,}")
    print(f"   Initial Error Rate: {initial_error_rate:.2f}%")
    
    # Apply Safety Net Corrections
    y_pred_safe = y_pred.copy()
    corrections_critical = 0
    corrections_high = 0
    
    if cgm_values is not None:
        for i in range(len(y_pred_safe)):
            if y_pred_safe[i] < DOSE_THRESHOLD:
                if cgm_values[i] > CGM_CRITICAL:
                    y_pred_safe[i] = 2.0
                    corrections_critical += 1
                elif cgm_values[i] > CGM_HIGH:
                    y_pred_safe[i] = 0.5
                    corrections_high += 1
    
    # Recalculate missed doses after safety net
    missed_after = ((y_test > DOSE_THRESHOLD) & (y_pred_safe < DOSE_THRESHOLD)).sum()
    final_error_rate = missed_after / pos_mask.sum() * 100 if pos_mask.sum() > 0 else 0
    
    print(f"\n🛡️ Safety Net Corrections:")
    print(f"   CGM > {CGM_CRITICAL} → Dose=2.0: {corrections_critical:,} corrections")
    print(f"   CGM > {CGM_HIGH} → Dose=0.5: {corrections_high:,} corrections")
    print(f"   Total corrections: {corrections_critical + corrections_high:,}")
    
    print(f"\n" + "=" * 50)
    print(f"📋 SAFETY NET REPORT")
    print(f"=" * 50)
    print(f"   ❌ Initial Error Rate:  {initial_error_rate:.2f}% ({n_missed_initial:,} missed)")
    print(f"   ✅ Final Safety Rate:   {final_error_rate:.2f}% ({missed_after:,} missed)")
    print(f"   📉 Reduction:           {initial_error_rate - final_error_rate:.2f}%")
    
    # Final metrics with safety net
    safe_mae = mean_absolute_error(y_test, y_pred_safe)
    print(f"\n   MAE (with safety): {safe_mae:.4f}")
    
else:
    print("\n⚠️ No target labels - predictions only")

## 📈 Visualizations

In [ ]:
if has_target:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # 1. Scatter: True vs Pred (positive only)
    ax1 = axes[0]
    pos = y_test > 0
    ax1.scatter(y_test[pos], y_pred[pos], alpha=0.3, s=10, c='steelblue')
    ax1.plot([0, y_test[pos].max()], [0, y_test[pos].max()], 'r--', lw=2, label='Perfect')
    ax1.set_xlabel('True Bolus')
    ax1.set_ylabel('Predicted Bolus')
    ax1.set_title(f'True vs Predicted (Positive Cases, n={pos.sum():,})')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 2. Feature Importance
    ax2 = axes[1]
    importance = model.get_score(importance_type='gain')
    imp_df = pd.DataFrame({'Feature': list(importance.keys()), 'Gain': list(importance.values())})
    imp_df = imp_df.sort_values('Gain', ascending=True)
    ax2.barh(imp_df['Feature'], imp_df['Gain'], color='coral')
    ax2.set_xlabel('Gain')
    ax2.set_title('Feature Importance')
    ax2.grid(True, alpha=0.3, axis='x')
    
    plt.tight_layout()
    plt.show()

print("\n" + "=" * 60)
print("🎉 PIPELINE COMPLETED SUCCESSFULLY!")
print("=" * 60)